# 01 — Data Inspection

## What this notebook does
1. **Setup** — mount Google Drive, define project paths, create the required folder structure, and check that the raw files used in the final analysis exist
2. **Inspect raw files** — inspect each required Excel file directly to identify metadata, header rows, data start rows, and relevant columns
3. **Document import decisions** — produce the import-decision table used by notebook 02
4. **Create quarterly skeleton** — define the master quarterly time structure required by the final TFM pipeline

## Final empirical scope
Baseline system: `[GPR_origin, Brent, Gas_TTF, HICP, IPI, CLIFS]`

GDP is retained for the descriptive GDP-vs-IPI comparison used to motivate the baseline activity indicator.

The master skeleton runs from **1997Q1 to 2025Q4**. The first observation is retained to construct quarterly log differences; the transformed estimation sample begins in **1997Q2**.


## Block A — Setup

In [9]:
# ============================================================
# PROJECT SETUP
# ============================================================
import sys
from pathlib import Path

# Google Colab: use the project stored in Google Drive
try:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/TFM_GitHub")

# Local execution: repository root or notebooks/ directory
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()

    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

# Check that the project structure is available
if not (PROJECT_ROOT / "src" / "paths.py").exists():
    raise FileNotFoundError(
        f"Could not find src/paths.py under: {PROJECT_ROOT}"
    )

# Make the project modules importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import *

print(f"Project root: {PROJECT_ROOT}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/TFM_GitHub


In [10]:
from pathlib import Path
import pandas as pd

print(f"pandas {pd.__version__}")
try:
    import openpyxl
    print(f"openpyxl {openpyxl.__version__}")
except ImportError:
    print("openpyxl not found — install with: pip install openpyxl")

pandas 2.2.3
openpyxl 3.1.5


In [11]:
# ============================================================
# A1. PROJECT PATHS
# ============================================================
# Paths are defined centrally in src/paths.py.

ensure_output_directories()

print("Paths defined.")
print(f"  Project:   {PROJECT_ROOT}")
print(f"  Raw data:  {RAW_DATA_DIR}")
print(f"  Processed: {PROCESSED_DATA_DIR}")

Paths defined.
  Project:   /content/drive/MyDrive/TFM_GitHub
  Raw data:  /content/drive/MyDrive/TFM_GitHub/data/raw
  Processed: /content/drive/MyDrive/TFM_GitHub/data/processed


In [12]:
# ============================================================
# A2. CREATE REQUIRED FOLDER STRUCTURE
# ============================================================

required_raw_folders = [
    RAW_DATA_DIR / "gpr",
    RAW_DATA_DIR / "energy",
    RAW_DATA_DIR / "hicp",
    RAW_DATA_DIR / "ipi",
    RAW_DATA_DIR / "gdp",
    RAW_DATA_DIR / "clifs",
]

for folder in required_raw_folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Required folder structure checked/created.")


Required folder structure checked/created.


In [13]:
# ============================================================
# A3. CHECK REQUIRED RAW DATA FILES
# ============================================================
# RAW_FILES is defined centrally in src/paths.py.

REQUIRED_FILE_KEYS = ["gpr", "energy", "hicp", "ipi", "gdp", "clifs"]
required_raw_files = {key: RAW_FILES[key] for key in REQUIRED_FILE_KEYS}

records = []
for key, path in required_raw_files.items():
    records.append({
        "file_key": key,
        "file_path": str(path),
        "exists": path.exists(),
        "file_extension": path.suffix.lower() if path.exists() else None,
    })

file_check = pd.DataFrame(records)
missing = file_check[~file_check["exists"]]

if not missing.empty:
    print("Missing files detected:")
    print(missing[["file_key", "file_path"]].to_string(index=False))
else:
    print("All required raw files found.")

file_check


All required raw files found.


,file_key,file_path,exists,file_extension
0,gpr,/content/drive/MyDrive/TFM_GitHub/data/raw/gpr...,True,.xlsx
1,energy,/content/drive/MyDrive/TFM_GitHub/data/raw/ene...,True,.xlsx
2,hicp,/content/drive/MyDrive/TFM_GitHub/data/raw/hic...,True,.xlsx
3,ipi,/content/drive/MyDrive/TFM_GitHub/data/raw/ipi...,True,.xlsx
4,gdp,/content/drive/MyDrive/TFM_GitHub/data/raw/gdp...,True,.xlsx
5,clifs,/content/drive/MyDrive/TFM_GitHub/data/raw/cli...,True,.xlsx


In [14]:
# ============================================================
# A4. LIST SHEETS IN EACH RAW FILE
# ============================================================

sheet_records = []

for key, path in required_raw_files.items():
    if not path.exists():
        print(f"Skipping {key}: file not found.")
        continue
    excel_file = pd.ExcelFile(path)
    for sheet in excel_file.sheet_names:
        sheet_records.append({
            "file_key"  : key,
            "file_name" : path.name,
            "sheet_name": sheet,
        })

excel_sheets = pd.DataFrame(sheet_records)
excel_sheets

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,file_key,file_name,sheet_name
0,gpr,GPR_DBGlobal_Bilateral.xlsx,Legend
1,gpr,GPR_DBGlobal_Bilateral.xlsx,Italy
2,gpr,GPR_DBGlobal_Bilateral.xlsx,Germany
3,gpr,GPR_DBGlobal_Bilateral.xlsx,France
4,gpr,GPR_DBGlobal_Bilateral.xlsx,Spain
5,gpr,GPR_DBGlobal_Bilateral.xlsx,Poland
6,gpr,GPR_DBGlobal_Bilateral.xlsx,Ireland
7,gpr,GPR_DBGlobal_Bilateral.xlsx,Austria
8,gpr,GPR_DBGlobal_Bilateral.xlsx,Portugal
9,gpr,GPR_DBGlobal_Bilateral.xlsx,Belgium


## Block B — Raw File Inspection

Each raw file is inspected directly, without going through any intermediate file.
The goal is to identify:
- where metadata ends and the data table begins
- exact header row positions
- which columns to use for each variable

In [15]:
# ============================================================
# HELPER: inspect_raw_sheet
# ============================================================
# Reads the first N rows of a sheet without assuming headers.
# Adds Excel-style row numbering for easy cross-reference.

def inspect_raw_sheet(file_path, sheet_name, nrows=30):
    print("\n" + "=" * 90)
    print(f"FILE:  {file_path.name}")
    print(f"SHEET: {sheet_name}")
    print("=" * 90)

    raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None, nrows=nrows)

    raw_display = raw.copy()
    raw_display.insert(0, "excel_row", range(1, len(raw_display) + 1))

    print(f"Preview: {raw.shape[0]} rows × {raw.shape[1]} columns")

    with pd.option_context("display.max_columns", 40, "display.max_colwidth", 100):
        display(raw_display)

    non_empty = raw_display.dropna(how="all", subset=raw_display.columns[1:])
    print("\nNon-empty rows only:")
    with pd.option_context("display.max_columns", 40, "display.max_colwidth", 100):
        display(non_empty)

    return raw

In [16]:
# ============================================================
# B1. GPR — BILATERAL INDEX BY COUNTRY
# ============================================================
# Source: Alonso-Alvarez et al. (2025, WP 2522, Banco de España)
# One sheet per country; columns are geopolitical origins.
# Expected frequency: quarterly (native, no aggregation needed).
# Target origins: Russia, China, MENA (Middle East), Western Bloc.

gpr_path = RAW_FILES["gpr"]

gpr_excel = pd.ExcelFile(gpr_path)
print("Available GPR sheets:", gpr_excel.sheet_names)

missing_sheets = [c for c in TARGET_COUNTRIES if c not in gpr_excel.sheet_names]
if missing_sheets:
    raise ValueError(f"Missing country sheets: {missing_sheets}")
print("All target country sheets present.")

Available GPR sheets: ['Legend', 'Italy', 'Germany', 'France', 'Spain', 'Poland', 'Ireland', 'Austria', 'Portugal', 'Belgium', 'Japan', 'United States', 'China', 'Switzerland', 'India', 'Brazil', 'Thailand', 'Philippines', 'Mexico', 'Argentina', 'Nigeria', 'Canada', 'Pakistan', 'Singapore', 'Malaysia', 'Colombia', 'Chile', 'New Zealand', 'Algeria', 'Qatar', 'Russia', 'Australia', 'Iran', 'Egypt', 'United Kingdom']
All target country sheets present.


In [17]:
# Inspect each country sheet
for country in TARGET_COUNTRIES:
    raw = inspect_raw_sheet(gpr_path, sheet_name=country, nrows=10)

    # Identify candidate origin columns from header row
    header_values = raw.iloc[0, :].tolist()
    header_df = pd.DataFrame({
        "python_col_index" : range(len(header_values)),
        "excel_col_number" : range(1, len(header_values) + 1),
        "column_name"      : header_values,
    })

    keywords = ["russia", "china", "mena", "middle", "western", "west"]
    candidates = header_df[
        header_df["column_name"]
        .astype(str).str.lower()
        .apply(lambda x: any(kw in x for kw in keywords))
    ]

    print("\nCandidate origin columns:")
    display(candidates)


FILE:  GPR_DBGlobal_Bilateral.xlsx
SHEET: Germany
Preview: 10 rows × 15 columns


,excel_row,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,1,Date,Russia,Russia_ext,China,China_ext,Middle East,Middle East_ext,Western Bloc,Western Bloc_ext,Tot,Ext,Russia_num,China_num,Middle East_num,Western Bloc_num
1,2,1997-01-01 00:00:00,9.474874,0.045455,0,0,38.046743,0.287879,24.595479,0.242424,11584,66,3,0,19,16
2,3,1997-04-01 00:00:00,22.084295,0.076923,33.970353,0.034965,45.825273,0.251748,44.950218,0.321678,18223,143,11,5,36,46
3,4,1997-07-01 00:00:00,16.060424,0.051724,18.116527,0.017241,56.571306,0.287356,52.981891,0.350575,20502,174,9,3,50,61
4,5,1997-10-01 00:00:00,27.573389,0.083871,21.533135,0.019355,91.44657,0.43871,62.97378,0.393548,17249,155,13,3,68,61
5,6,1998-01-01 00:00:00,15.280629,0.050633,8.61845,0.008439,88.003144,0.459916,65.697519,0.447257,28731,237,12,2,109,106
6,7,1998-04-01 00:00:00,19.425102,0.064846,38.057617,0.037543,43.430638,0.228669,58.718484,0.40273,35785,293,19,11,67,118
7,8,1998-07-01 00:00:00,15.389723,0.058824,5.786654,0.006536,47.703766,0.287582,54.098443,0.424837,42791,306,18,2,88,130
8,9,1998-10-01 00:00:00,12.041353,0.053892,13.582923,0.017964,70.408095,0.497006,51.11938,0.47006,54690,334,18,6,166,157
9,10,1999-01-01 00:00:00,36.391838,0.108861,0,0,54.732768,0.258228,60.14112,0.36962,43229,395,43,0,102,146



Non-empty rows only:


,excel_row,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,1,Date,Russia,Russia_ext,China,China_ext,Middle East,Middle East_ext,Western Bloc,Western Bloc_ext,Tot,Ext,Russia_num,China_num,Middle East_num,Western Bloc_num
1,2,1997-01-01 00:00:00,9.474874,0.045455,0,0,38.046743,0.287879,24.595479,0.242424,11584,66,3,0,19,16
2,3,1997-04-01 00:00:00,22.084295,0.076923,33.970353,0.034965,45.825273,0.251748,44.950218,0.321678,18223,143,11,5,36,46
3,4,1997-07-01 00:00:00,16.060424,0.051724,18.116527,0.017241,56.571306,0.287356,52.981891,0.350575,20502,174,9,3,50,61
4,5,1997-10-01 00:00:00,27.573389,0.083871,21.533135,0.019355,91.44657,0.43871,62.97378,0.393548,17249,155,13,3,68,61
5,6,1998-01-01 00:00:00,15.280629,0.050633,8.61845,0.008439,88.003144,0.459916,65.697519,0.447257,28731,237,12,2,109,106
6,7,1998-04-01 00:00:00,19.425102,0.064846,38.057617,0.037543,43.430638,0.228669,58.718484,0.40273,35785,293,19,11,67,118
7,8,1998-07-01 00:00:00,15.389723,0.058824,5.786654,0.006536,47.703766,0.287582,54.098443,0.424837,42791,306,18,2,88,130
8,9,1998-10-01 00:00:00,12.041353,0.053892,13.582923,0.017964,70.408095,0.497006,51.11938,0.47006,54690,334,18,6,166,157
9,10,1999-01-01 00:00:00,36.391838,0.108861,0,0,54.732768,0.258228,60.14112,0.36962,43229,395,43,0,102,146



Candidate origin columns:


...

,excel_row,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,1,Date,Russia,Russia_ext,China,China_ext,Middle East,Middle East_ext,Western Bloc,Western Bloc_ext,Tot,Ext,Russia_num,China_num,Middle East_num,Western Bloc_num
1,2,1996-01-01 00:00:00,142.160188,0.177606,202.929821,0.073359,83.252867,0.254826,106.877525,0.455598,21010,259,46,19,66,118
2,3,1996-04-01 00:00:00,115.963434,0.153527,129.978854,0.049793,93.38504,0.302905,73.48413,0.33195,20717,241,37,12,73,80
3,4,1996-07-01 00:00:00,146.650042,0.152542,33.788044,0.010169,119.714648,0.305085,114.613334,0.40678,19924,295,45,3,90,120
4,5,1996-10-01 00:00:00,61.194791,0.099602,33.838145,0.015936,64.941588,0.258964,48.782896,0.270916,26526,251,25,4,65,68
5,6,1997-01-01 00:00:00,47.502121,0.07563,18.240746,0.008403,51.70313,0.201681,58.781181,0.319328,24604,238,18,2,48,76
6,7,1997-04-01 00:00:00,69.328638,0.119601,73.210768,0.036545,33.799768,0.142857,72.808838,0.428571,33716,301,36,11,43,129
7,8,1997-07-01 00:00:00,43.402487,0.070845,28.845853,0.013624,85.169949,0.340599,65.06945,0.362398,38896,367,26,5,125,133
8,9,1997-10-01 00:00:00,67.268028,0.106522,61.677687,0.028261,109.265317,0.423913,97.769437,0.528261,47297,460,49,13,195,243
9,10,1998-01-01 00:00:00,62.926946,0.085106,4.530723,0.001773,137.519298,0.455674,104.123542,0.480496,49528,564,48,1,257,271



Non-empty rows only:


,excel_row,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,1,Date,Russia,Russia_ext,China,China_ext,Middle East,Middle East_ext,Western Bloc,Western Bloc_ext,Tot,Ext,Russia_num,China_num,Middle East_num,Western Bloc_num
1,2,1996-01-01 00:00:00,142.160188,0.177606,202.929821,0.073359,83.252867,0.254826,106.877525,0.455598,21010,259,46,19,66,118
2,3,1996-04-01 00:00:00,115.963434,0.153527,129.978854,0.049793,93.38504,0.302905,73.48413,0.33195,20717,241,37,12,73,80
3,4,1996-07-01 00:00:00,146.650042,0.152542,33.788044,0.010169,119.714648,0.305085,114.613334,0.40678,19924,295,45,3,90,120
4,5,1996-10-01 00:00:00,61.194791,0.099602,33.838145,0.015936,64.941588,0.258964,48.782896,0.270916,26526,251,25,4,65,68
5,6,1997-01-01 00:00:00,47.502121,0.07563,18.240746,0.008403,51.70313,0.201681,58.781181,0.319328,24604,238,18,2,48,76
6,7,1997-04-01 00:00:00,69.328638,0.119601,73.210768,0.036545,33.799768,0.142857,72.808838,0.428571,33716,301,36,11,43,129
7,8,1997-07-01 00:00:00,43.402487,0.070845,28.845853,0.013624,85.169949,0.340599,65.06945,0.362398,38896,367,26,5,125,133
8,9,1997-10-01 00:00:00,67.268028,0.106522,61.677687,0.028261,109.265317,0.423913,97.769437,0.528261,47297,460,49,13,195,243
9,10,1998-01-01 00:00:00,62.926946,0.085106,4.530723,0.001773,137.519298,0.455674,104.123542,0.480496,49528,564,48,1,257,271



Candidate origin columns:


,python_col_index,excel_col_number,column_name
1,1,2,Russia
2,2,3,Russia_ext
3,3,4,China
4,4,5,China_ext
5,5,6,Middle East
6,6,7,Middle East_ext
7,7,8,Western Bloc
8,8,9,Western Bloc_ext
11,11,12,Russia_num
12,12,13,China_num


In [18]:
# ============================================================
# B2. ENERGY PRICES — BRENT AND GAS TTF
# ============================================================
# Source: World Bank Pink Sheet
# Sheet: Monthly Prices
# Frequency: monthly (will be aggregated to quarterly in notebook 02)
# Target columns: Crude oil Brent, Natural gas Europe

energy_path = RAW_FILES["energy"]

energy_raw = inspect_raw_sheet(energy_path, sheet_name="Monthly Prices", nrows=25)

# Transpose header rows for easier column identification
energy_header_check = pd.DataFrame({
    "python_col_index"  : energy_raw.columns,
    "excel_col_number"  : [i + 1 for i in energy_raw.columns],
    "row_5_name"        : energy_raw.iloc[4, :].values,
    "row_6_unit"        : energy_raw.iloc[5, :].values,
})

print("\nEnergy header rows (transposed):")
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 120):
    display(energy_header_check)

# Search candidate columns
keywords = ["brent", "gas", "natural", "europe", "ttf", "lng"]
energy_candidates = energy_header_check[
    (energy_header_check["row_5_name"].astype(str).str.lower()
     + " " + energy_header_check["row_6_unit"].astype(str).str.lower())
    .apply(lambda x: any(kw in x for kw in keywords))
]

print("\nCandidate Brent / Gas columns:")
display(energy_candidates)


FILE:  Brent_and_Gas_Prices.xlsx
SHEET: Monthly Prices
Preview: 25 rows × 72 columns


,excel_row,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,...,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71
0,1,World Bank Commodity Price Data (The Pink Sheet),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,"monthly prices in nominal US dollars, 1960 to present",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,(monthly series are available only in nominal US dollars),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,"Updated on June 02, 2026",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,NaN,"Crude oil, average","Crude oil, Brent","Crude oil, Dubai","Crude oil, WTI","Coal, Australian","Coal, South African **","Natural gas, US","Natural gas, Europe","Liquefied natural gas, Japan",Natural gas index,Cocoa,"Coffee, Arabica","Coffee, Robusta","Tea, avg 3 auctions","Tea, Colombo","Tea, Kolkata","Tea, Mombasa",Coconut oil,...,"Sawnwood, Malaysian",Plywood,"Cotton, A Index","Rubber, TSR20 **","Rubber, RSS3",Phosphate rock,DAP,TSP,Urea,Potassium chloride **,Aluminum,"Iron ore, cfr spot",Copper,Lead,Tin,Nickel,Zinc,Gold,Platinum,Silver
5,6,NaN,($/bbl),($/bbl),($/bbl),($/bbl),($/mt),($/mt),($/mmbtu),($/mmbtu),($/mmbtu),(2010=100),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/mt),...,($/cubic meter),(cents/sheet),($/kg),($/kg),($/kg),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/dmtu),($/mt),($/mt),($/mt),($/mt),($/mt),($/troy oz),($/troy oz),($/troy oz)
6,7,1960M01,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.63,0.94,0.7,1.03,0.93,1.12,1.04,390,...,149.17,…,0.65,…,0.82,13,...,53,42.25,28.5,511.47,11.42,715.4,206.1,2180.4,1631,260.8,35.27,83.5,0.91
7,8,1960M02,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.61,0.95,0.69,1.03,0.93,1.12,1.04,379,...,149.17,…,0.65,…,0.83,13,...,53,42.25,28.5,511.47,11.42,728.19,203.7,2180.4,1631,244.9,35.27,83.5,0.91
8,9,1960M03,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.58,0.93,0.69,1.03,0.93,1.12,1.04,361,...,149.17,…,0.65,…,0.86,13,...,53,42.25,28.5,511.47,11.42,684.94,210.3,2173.8,1631,248.7,35.27,83.5,0.91
9,10,1960M04,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.6,0.93,0.68,1.03,0.93,1.12,1.04,338,...,149.17,…,0.64,…,0.86,13,...,53,42.25,28.5,511.47,11.42,723.11,213.6,2178.2,1631,254.6,35.27,83.5,0.91



Non-empty rows only:


,excel_row,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,...,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71
0,1,World Bank Commodity Price Data (The Pink Sheet),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,"monthly prices in nominal US dollars, 1960 to present",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,(monthly series are available only in nominal US dollars),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,"Updated on June 02, 2026",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,NaN,"Crude oil, average","Crude oil, Brent","Crude oil, Dubai","Crude oil, WTI","Coal, Australian","Coal, South African **","Natural gas, US","Natural gas, Europe","Liquefied natural gas, Japan",Natural gas index,Cocoa,"Coffee, Arabica","Coffee, Robusta","Tea, avg 3 auctions","Tea, Colombo","Tea, Kolkata","Tea, Mombasa",Coconut oil,...,"Sawnwood, Malaysian",Plywood,"Cotton, A Index","Rubber, TSR20 **","Rubber, RSS3",Phosphate rock,DAP,TSP,Urea,Potassium chloride **,Aluminum,"Iron ore, cfr spot",Copper,Lead,Tin,Nickel,Zinc,Gold,Platinum,Silver
5,6,NaN,($/bbl),($/bbl),($/bbl),($/bbl),($/mt),($/mt),($/mmbtu),($/mmbtu),($/mmbtu),(2010=100),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/mt),...,($/cubic meter),(cents/sheet),($/kg),($/kg),($/kg),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/dmtu),($/mt),($/mt),($/mt),($/mt),($/mt),($/troy oz),($/troy oz),($/troy oz)
6,7,1960M01,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.63,0.94,0.7,1.03,0.93,1.12,1.04,390,...,149.17,…,0.65,…,0.82,13,...,53,42.25,28.5,511.47,11.42,715.4,206.1,2180.4,1631,260.8,35.27,83.5,0.91
7,8,1960M02,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.61,0.95,0.69,1.03,0.93,1.12,1.04,379,...,149.17,…,0.65,…,0.83,13,...,53,42.25,28.5,511.47,11.42,728.19,203.7,2180.4,1631,244.9,35.27,83.5,0.91
8,9,1960M03,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.58,0.93,0.69,1.03,0.93,1.12,1.04,361,...,149.17,…,0.65,…,0.86,13,...,53,42.25,28.5,511.47,11.42,684.94,210.3,2173.8,1631,248.7,35.27,83.5,0.91
9,10,1960M04,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.6,0.93,0.68,1.03,0.93,1.12,1.04,338,...,149.17,…,0.64,…,0.86,13,...,53,42.25,28.5,511.47,11.42,723.11,213.6,2178.2,1631,254.6,35.27,83.5,0.91



Energy header rows (transposed):


,python_col_index,excel_col_number,row_5_name,row_6_unit
0,0,1,NaN,NaN
1,1,2,"Crude oil, average",($/bbl)
2,2,3,"Crude oil, Brent",($/bbl)
3,3,4,"Crude oil, Dubai",($/bbl)
4,4,5,"Crude oil, WTI",($/bbl)
5,5,6,"Coal, Australian",($/mt)
6,6,7,"Coal, South African **",($/mt)
7,7,8,"Natural gas, US",($/mmbtu)
8,8,9,"Natural gas, Europe",($/mmbtu)
9,9,10,"Liquefied natural gas, Japan",($/mmbtu)



Candidate Brent / Gas columns:


,python_col_index,excel_col_number,row_5_name,row_6_unit
2,2,3,"Crude oil, Brent",($/bbl)
7,7,8,"Natural gas, US",($/mmbtu)
8,8,9,"Natural gas, Europe",($/mmbtu)
9,9,10,"Liquefied natural gas, Japan",($/mmbtu)
10,10,11,Natural gas index,(2010=100)
38,38,39,"Banana, Europe",($/kg)


In [19]:
# ============================================================
# B4. HICP — HEADLINE ALL-ITEMS
# ============================================================
# Source: Eurostat via ECB
# Frequency: monthly (will be aggregated to quarterly in notebook 02)
# We use Sheet 1 (all-items HICP) for the BVAR.

hicp_path = RAW_FILES["hicp"]

inspect_raw_sheet(hicp_path, sheet_name="Sheet 1", nrows=15);


FILE:  HICP_CoreHICP_EnergyHICP.xlsx
SHEET: Sheet 1
Preview: 15 rows × 361 columns


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,excel_row,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,...,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360
0,1,Data extracted on 04/06/2026 12:28:25 from [ESTAT],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Dataset:,HICP - monthly data (index) (1996-2025) [prc_hicp_midx__custom_21719578],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Last updated:,06/02/2026 23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Time frequency,NaN,Monthly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,Unit of measure,NaN,"Index, 2015=100",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,Classification of individual consumption by purpose (COICOP),NaN,All-items HICP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,TIME,1996-01,1996-02,1996-03,1996-04,1996-05,1996-06,1996-07,1996-08,1996-09,1996-10,1996-11,1996-12,1997-01,1997-02,1997-03,1997-04,1997-05,1997-06,...,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
9,10,GEO (Labels),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Non-empty rows only:


,excel_row,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,...,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360
0,1,Data extracted on 04/06/2026 12:28:25 from [ESTAT],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Dataset:,HICP - monthly data (index) (1996-2025) [prc_hicp_midx__custom_21719578],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Last updated:,06/02/2026 23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Time frequency,NaN,Monthly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,Unit of measure,NaN,"Index, 2015=100",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,Classification of individual consumption by purpose (COICOP),NaN,All-items HICP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,TIME,1996-01,1996-02,1996-03,1996-04,1996-05,1996-06,1996-07,1996-08,1996-09,1996-10,1996-11,1996-12,1997-01,1997-02,1997-03,1997-04,1997-05,1997-06,...,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
9,10,GEO (Labels),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,11,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA15-2008, EA16-2009, EA17-2011, EA18-2014, EA19-201...",70.97,71.29,71.54,71.66,71.83,71.84,71.84,71.8,71.93,72,72.01,72.19,72.42,72.59,72.65,72.6,72.8,72.83,...,126.31,126.58,126.54,126.72,126.6,127.03,126.62,127.07,126.72,127.26,128.04,128.77,128.7,129.09,129.12,129.31,129.42,129.7,129.33,129.56
11,12,Germany,75.1,75.6,75.7,75.6,75.7,75.8,76,75.9,75.9,75.8,75.7,76,76.5,76.8,76.7,76.5,76.8,76.9,...,129.1,129.3,130,129.8,129.7,130.2,129.3,130.2,129.9,130.5,131,131.6,131.8,131.9,132.4,132.5,132.8,133.2,132.6,132.8


In [20]:
# ============================================================
# B5. GDP
# ============================================================
# Source: Eurostat
# Frequency: quarterly (no aggregation needed)
# GDP is retained for the descriptive GDP-vs-IPI comparison.

gdp_path = RAW_FILES["gdp"]
inspect_raw_sheet(gdp_path, sheet_name="Sheet 1", nrows=20);


FILE:  GDP.xlsx
SHEET: Sheet 1
Preview: 18 rows × 146 columns


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,excel_row,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,...,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145
0,1,Data extracted on 04/06/2026 13:03:09 from [ESTAT],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Dataset:,"Gross domestic product (GDP) and main components (output, expenditure and income) - quarterly da...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Last updated:,03/06/2026 23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Time frequency,NaN,Quarterly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,Unit of measure,NaN,"Chain linked volumes, index 2015=100",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,Seasonal adjustment,NaN,Seasonally and calendar adjusted data,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,National accounts indicator (ESA 2010),NaN,Gross domestic product at market prices,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,TIME,1990-Q1,1990-Q2,1990-Q3,1990-Q4,1991-Q1,1991-Q2,1991-Q3,1991-Q4,1992-Q1,1992-Q2,1992-Q3,1992-Q4,1993-Q1,1993-Q2,1993-Q3,1993-Q4,1994-Q1,1994-Q2,...,2021-Q2,2021-Q3,2021-Q4,2022-Q1,2022-Q2,2022-Q3,2022-Q4,2023-Q1,2023-Q2,2023-Q3,2023-Q4,2024-Q1,2024-Q2,2024-Q3,2024-Q4,2025-Q1,2025-Q2,2025-Q3,2025-Q4,2026-Q1



Non-empty rows only:


,excel_row,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,...,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145
0,1,Data extracted on 04/06/2026 13:03:09 from [ESTAT],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Dataset:,"Gross domestic product (GDP) and main components (output, expenditure and income) - quarterly da...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Last updated:,03/06/2026 23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Time frequency,NaN,Quarterly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,Unit of measure,NaN,"Chain linked volumes, index 2015=100",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,Seasonal adjustment,NaN,Seasonally and calendar adjusted data,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,National accounts indicator (ESA 2010),NaN,Gross domestic product at market prices,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,TIME,1990-Q1,1990-Q2,1990-Q3,1990-Q4,1991-Q1,1991-Q2,1991-Q3,1991-Q4,1992-Q1,1992-Q2,1992-Q3,1992-Q4,1993-Q1,1993-Q2,1993-Q3,1993-Q4,1994-Q1,1994-Q2,...,2021-Q2,2021-Q3,2021-Q4,2022-Q1,2022-Q2,2022-Q3,2022-Q4,2023-Q1,2023-Q2,2023-Q3,2023-Q4,2024-Q1,2024-Q2,2024-Q3,2024-Q4,2025-Q1,2025-Q2,2025-Q3,2025-Q4,2026-Q1
10,11,GEO (Labels),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,12,Germany,:,:,:,:,72.909,72.559,72.446,73.486,74.497,74.034,73.859,73.653,73.069,73.079,73.511,73.45,74.487,74.899,...,107.159,107.252,107.828,108.559,108.724,109.043,108.662,108.15,108.068,108.068,107.769,107.653,107.375,107.396,107.591,107.993,107.777,107.735,107.993,108.364


In [21]:
# ============================================================
# B6. IPI — INDUSTRIAL PRODUCTION INDEX
# ============================================================
# Source: Eurostat
# Frequency: monthly (will be aggregated to quarterly in notebook 02)
# IPI is the activity variable used in the baseline specification.

ipi_path = RAW_FILES["ipi"]

inspect_raw_sheet(ipi_path, sheet_name="Sheet 1", nrows=20);


FILE:  IPI.xlsx
SHEET: Sheet 1
Preview: 19 rows × 437 columns

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,excel_row,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,...,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436
0,1,Data extracted on 04/06/2026 12:59:54 from [ESTAT],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Dataset:,Production in industry - monthly data [sts_inpr_m__custom_21720139],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Last updated:,04/06/2026 11:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Time frequency,NaN,Monthly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,Business trend indicator,NaN,Production (volume),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,Statistical classification of economic activities in the European Community (NACE Rev. 2),NaN,"Mining and quarrying; manufacturing; electricity, gas, steam and air conditioning supply",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,Seasonal adjustment,NaN,Seasonally and calendar adjusted data,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,Unit of measure,NaN,"Index, 2021=100",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Non-empty rows only:


,excel_row,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,...,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436
0,1,Data extracted on 04/06/2026 12:59:54 from [ESTAT],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Dataset:,Production in industry - monthly data [sts_inpr_m__custom_21720139],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Last updated:,04/06/2026 11:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Time frequency,NaN,Monthly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,Business trend indicator,NaN,Production (volume),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,Statistical classification of economic activities in the European Community (NACE Rev. 2),NaN,"Mining and quarrying; manufacturing; electricity, gas, steam and air conditioning supply",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,Seasonal adjustment,NaN,Seasonally and calendar adjusted data,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,Unit of measure,NaN,"Index, 2021=100",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,11,TIME,1990-01,1990-02,1990-03,1990-04,1990-05,1990-06,1990-07,1990-08,1990-09,1990-10,1990-11,1990-12,1991-01,1991-02,1991-03,1991-04,1991-05,1991-06,...,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04
11,12,GEO (Labels),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# ============================================================
# B7. CLIFS — COMPOSITE INDICATOR OF SYSTEMIC STRESS
# ============================================================
# Source: ECB
# Sheet: DATA
# Frequency: monthly (will be aggregated to quarterly in notebook 02)
# Target columns: Germany, France, Italy, Spain

clifs_path = RAW_FILES["clifs"]
clifs_raw = inspect_raw_sheet(clifs_path, sheet_name="DATA", nrows=30)

print("\nRows around data start (excel rows 14–22):")
clifs_display = clifs_raw.copy()
clifs_display.insert(0, "excel_row", range(1, len(clifs_display) + 1))
with pd.option_context("display.max_columns", 30, "display.max_colwidth", 100):
    display(clifs_display.iloc[13:22, :]);


FILE:  CLIFS.xlsx
SHEET: DATA
Preview: 30 rows × 6 columns


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,excel_row,0,1,2,3,4,5
0,1,ECB Data Portal,NaN,NaN,NaN,NaN,NaN
1,2,Downloaded on: 2026-06-04 13:05:12,NaN,NaN,NaN,NaN,NaN
2,3,NaN,NaN,NaN,NaN,NaN,NaN
3,4,TITLE,NaN,Country Level Index of Financial Stress (CLIFS) Germany,Country Level Index of Financial Stress (CLIFS) Spain,Country Level Index of Financial Stress (CLIFS) France,Country Level Index of Financial Stress (CLIFS) Italy
4,5,SERIES KEY,NaN,CLIFS.M.DE._Z.4F.EC.CLIFS_CI.IDX,CLIFS.M.ES._Z.4F.EC.CLIFS_CI.IDX,CLIFS.M.FR._Z.4F.EC.CLIFS_CI.IDX,CLIFS.M.IT._Z.4F.EC.CLIFS_CI.IDX
5,6,TIME SERIES PAGE,NaN,https://data.ecb.europa.eu/data/datasets/CLIFS/CLIFS.M.DE._Z.4F.EC.CLIFS_CI.IDX,https://data.ecb.europa.eu/data/datasets/CLIFS/CLIFS.M.ES._Z.4F.EC.CLIFS_CI.IDX,https://data.ecb.europa.eu/data/datasets/CLIFS/CLIFS.M.FR._Z.4F.EC.CLIFS_CI.IDX,https://data.ecb.europa.eu/data/datasets/CLIFS/CLIFS.M.IT._Z.4F.EC.CLIFS_CI.IDX
6,7,GEOGRAPHICAL AREA,NaN,Germany (DE),Spain (ES),France (FR),Italy (IT)
7,8,FREQUENCY,NaN,Monthly,Monthly,Monthly,Monthly
8,9,DATASET,NaN,CLIFS,CLIFS,CLIFS,CLIFS
9,10,DATA SOURCE,NaN,ECB,ECB,ECB,ECB



Non-empty rows only:


,excel_row,0,1,2,3,4,5
0,1,ECB Data Portal,NaN,NaN,NaN,NaN,NaN
1,2,Downloaded on: 2026-06-04 13:05:12,NaN,NaN,NaN,NaN,NaN
3,4,TITLE,NaN,Country Level Index of Financial Stress (CLIFS) Germany,Country Level Index of Financial Stress (CLIFS) Spain,Country Level Index of Financial Stress (CLIFS) France,Country Level Index of Financial Stress (CLIFS) Italy
4,5,SERIES KEY,NaN,CLIFS.M.DE._Z.4F.EC.CLIFS_CI.IDX,CLIFS.M.ES._Z.4F.EC.CLIFS_CI.IDX,CLIFS.M.FR._Z.4F.EC.CLIFS_CI.IDX,CLIFS.M.IT._Z.4F.EC.CLIFS_CI.IDX
5,6,TIME SERIES PAGE,NaN,https://data.ecb.europa.eu/data/datasets/CLIFS/CLIFS.M.DE._Z.4F.EC.CLIFS_CI.IDX,https://data.ecb.europa.eu/data/datasets/CLIFS/CLIFS.M.ES._Z.4F.EC.CLIFS_CI.IDX,https://data.ecb.europa.eu/data/datasets/CLIFS/CLIFS.M.FR._Z.4F.EC.CLIFS_CI.IDX,https://data.ecb.europa.eu/data/datasets/CLIFS/CLIFS.M.IT._Z.4F.EC.CLIFS_CI.IDX
6,7,GEOGRAPHICAL AREA,NaN,Germany (DE),Spain (ES),France (FR),Italy (IT)
7,8,FREQUENCY,NaN,Monthly,Monthly,Monthly,Monthly
8,9,DATASET,NaN,CLIFS,CLIFS,CLIFS,CLIFS
9,10,DATA SOURCE,NaN,ECB,ECB,ECB,ECB
10,11,COLLECTION TYPE,NaN,Other (V),Other (V),Other (V),Other (V)



Rows around data start (excel rows 14–22):


,excel_row,0,1,2,3,4,5
13,14,NaN,NaN,NaN,NaN,NaN,NaN
14,15,DATE,TIME PERIOD,Country Level Index of Financial Stress (CLIFS) Germany (CLIFS.M.DE._Z.4F.EC.CLIFS_CI.IDX),Country Level Index of Financial Stress (CLIFS) Spain (CLIFS.M.ES._Z.4F.EC.CLIFS_CI.IDX),Country Level Index of Financial Stress (CLIFS) France (CLIFS.M.FR._Z.4F.EC.CLIFS_CI.IDX),Country Level Index of Financial Stress (CLIFS) Italy (CLIFS.M.IT._Z.4F.EC.CLIFS_CI.IDX)
15,16,1970-01-31,1970Jan,0.0694,0.0456,0.1307,0.0711
16,17,1970-02-28,1970Feb,0.0716,0.092,0.1031,0.0823
17,18,1970-03-31,1970Mar,0.1009,0.0621,0.0712,0.0437
18,19,1970-04-30,1970Apr,0.0958,0.0871,0.0661,0.0484
19,20,1970-05-31,1970May,0.1267,0.1096,0.1332,0.054
20,21,1970-06-30,1970Jun,0.12,0.0614,0.1192,0.0437
21,22,1970-07-31,1970Jul,0.1329,0.0454,0.0929,0.067


## Block C — Import Decision Table

This table is the output of the inspection above.
It records the exact import parameters that notebook 02 will use for each variable.
Update this table if you discover different header rows or column positions during inspection.

In [23]:
# ============================================================
# C1. BUILD IMPORT DECISION TABLE
# ============================================================

import_decisions = pd.DataFrame([
    {
        "variable_group"      : "GPR",
        "raw_file"            : "GPR_DBGlobal_Bilateral.xlsx",
        "folder"              : "data/raw/gpr",
        "sheet_to_use"        : "Germany, France, Italy, Spain (one sheet per country)",
        "excel_header_row"    : 1,
        "python_header_row"   : 0,
        "excel_data_start_row": 2,
        "original_frequency"  : "Quarterly",
        "columns_to_use"      : "Date, Russia, China, Middle East, Western Bloc",
        "treatment_in_nb02"   : "Select target countries and origins; rename Middle East → MENA",
    },
    {
        "variable_group"      : "Energy prices",
        "raw_file"            : "Brent_and_Gas_Prices.xlsx",
        "folder"              : "data/raw/energy",
        "sheet_to_use"        : "Monthly Prices",
        "excel_header_row"    : 5,
        "python_header_row"   : 4,
        "excel_data_start_row": 7,
        "original_frequency"  : "Monthly",
        "columns_to_use"      : "Date, Crude oil Brent, Natural gas Europe (TTF proxy)",
        "treatment_in_nb02"   : "Aggregate monthly USD energy-price series to quarterly means",
    },
    {
        "variable_group"      : "HICP (all-items headline)",
        "raw_file"            : "HICP_CoreHICP_EnergyHICP.xlsx",
        "folder"              : "data/raw/hicp",
        "sheet_to_use"        : "Sheet 1",
        "excel_header_row"    : 9,
        "python_header_row"   : 8,
        "excel_data_start_row": 11,
        "original_frequency"  : "Monthly",
        "columns_to_use"      : "TIME + Germany, France, Italy, Spain rows",
        "treatment_in_nb02"   : "Reshape wide → long; aggregate to quarterly mean",
    },
    {
        "variable_group"      : "GDP",
        "raw_file"            : "GDP.xlsx",
        "folder"              : "data/raw/gdp",
        "sheet_to_use"        : "Sheet 1",
        "excel_header_row"    : 10,
        "python_header_row"   : 9,
        "excel_data_start_row": 12,
        "original_frequency"  : "Quarterly",
        "columns_to_use"      : "TIME + Germany, France, Italy, Spain rows",
        "treatment_in_nb02"   : "Reshape wide → long; use quarterly series directly for the descriptive GDP-vs-IPI comparison.",
    },
    {
        "variable_group"      : "IPI (industrial production index)",
        "raw_file"            : "IPI.xlsx",
        "folder"              : "data/raw/ipi",
        "sheet_to_use"        : "Sheet 1",
        "excel_header_row"    : 11,
        "python_header_row"   : 10,
        "excel_data_start_row": 13,
        "original_frequency"  : "Monthly",
        "columns_to_use"      : "TIME + Germany, Spain, France, Italy rows",
        "treatment_in_nb02"   : "Replace ':' with NaN; reshape wide → long; aggregate to quarterly mean for the baseline specification.",
    },
    {
        "variable_group"      : "CLIFS",
        "raw_file"            : "CLIFS.xlsx",
        "folder"              : "data/raw/clifs",
        "sheet_to_use"        : "DATA",
        "excel_header_row"    : 15,
        "python_header_row"   : 14,
        "excel_data_start_row": 16,
        "original_frequency"  : "Monthly",
        "columns_to_use"      : "DATE, Germany, Spain, France, Italy",
        "treatment_in_nb02"   : "Rename columns; aggregate to quarterly mean",
    },
])

with pd.option_context("display.max_colwidth", 80, "display.max_rows", 20):
    display(import_decisions)

,variable_group,raw_file,folder,sheet_to_use,excel_header_row,python_header_row,excel_data_start_row,original_frequency,columns_to_use,treatment_in_nb02
0,GPR,GPR_DBGlobal_Bilateral.xlsx,data/raw/gpr,"Germany, France, Italy, Spain (one sheet per country)",1,0,2,Quarterly,"Date, Russia, China, Middle East, Western Bloc",Select target countries and origins; rename Middle East → MENA
1,Energy prices,Brent_and_Gas_Prices.xlsx,data/raw/energy,Monthly Prices,5,4,7,Monthly,"Date, Crude oil Brent, Natural gas Europe (TTF proxy)",Aggregate monthly USD energy-price series to quarterly means
2,HICP (all-items headline),HICP_CoreHICP_EnergyHICP.xlsx,data/raw/hicp,Sheet 1,9,8,11,Monthly,"TIME + Germany, France, Italy, Spain rows",Reshape wide → long; aggregate to quarterly mean
3,GDP,GDP.xlsx,data/raw/gdp,Sheet 1,10,9,12,Quarterly,"TIME + Germany, France, Italy, Spain rows",Reshape wide → long; use quarterly series directly for the descriptive GDP-v...
4,IPI (industrial production index),IPI.xlsx,data/raw/ipi,Sheet 1,11,10,13,Monthly,"TIME + Germany, Spain, France, Italy rows",Replace ':' with NaN; reshape wide → long; aggregate to quarterly mean for t...
5,CLIFS,CLIFS.xlsx,data/raw/clifs,DATA,15,14,16,Monthly,"DATE, Germany, Spain, France, Italy",Rename columns; aggregate to quarterly mean


In [24]:
# Save decision table
import_decisions.to_excel(IMPORT_DECISIONS_PATH, index=False)
print(f"Import decision table saved to: {IMPORT_DECISIONS_PATH}")


Import decision table saved to: /content/drive/MyDrive/TFM_GitHub/data/processed/01_import_decisions.xlsx


## Block D — Quarterly Master Skeleton

In [25]:
# ============================================================
# CHECK DATE RANGES FOR ALL RAW VARIABLES (CONSOLIDATED, FINAL)
# ============================================================
# Check the raw date coverage of the sources required by the final analysis.

import pandas as pd
import re

date_range_results = []

# ------------------------------------------------------------
# 1. LONG-FORMAT, STANDARD DATE (first column, going down rows)
# ------------------------------------------------------------
long_format_sources = {
    "CLIFS":   {"file_key": "clifs", "sheet": "DATA", "header": 14},
}

for label, cfg in long_format_sources.items():
    path = RAW_FILES[cfg["file_key"]]
    df = pd.read_excel(path, sheet_name=cfg["sheet"], header=cfg["header"])
    date_series = pd.to_datetime(df.iloc[:, 0], errors="coerce").dropna()

    date_range_results.append({
        "variable": label,
        "country": "N/A (single series)",
        "start": date_series.min(),
        "end": date_series.max(),
    })

# ------------------------------------------------------------
# 2. LONG-FORMAT, NON-STANDARD DATE (World Bank "YYYYMxx" text)
# ------------------------------------------------------------
def parse_worldbank_date(text):
    match = re.match(r"(\d{4})M(\d{2})", str(text))
    if match:
        year, month = match.groups()
        return pd.Timestamp(year=int(year), month=int(month), day=1)
    return pd.NaT

energy_path = RAW_FILES["energy"]
energy_df = pd.read_excel(energy_path, sheet_name="Monthly Prices", header=4)
energy_dates = energy_df.iloc[:, 0].apply(parse_worldbank_date).dropna()

date_range_results.append({
    "variable": "Energy (Brent/Gas)",
    "country": "N/A (single series)",
    "start": energy_dates.min(),
    "end": energy_dates.max(),
})

# ------------------------------------------------------------
# 3. GPR — long format, standard date, ONE SHEET PER COUNTRY
# ------------------------------------------------------------
gpr_path = RAW_FILES["gpr"]

for country in TARGET_COUNTRIES:
    df = pd.read_excel(gpr_path, sheet_name=country, header=0)
    date_series = pd.to_datetime(df.iloc[:, 0], errors="coerce").dropna()

    date_range_results.append({
        "variable": "GPR",
        "country": country,
        "start": date_series.min(),
        "end": date_series.max(),
    })

# ------------------------------------------------------------
# 4. WIDE-FORMAT SOURCES (dates as column headers, countries as rows)
# ------------------------------------------------------------
wide_format_sources = {
    "HICP": {"file_key": "hicp", "sheet": "Sheet 1", "header": 8},
    "GDP":  {"file_key": "gdp",  "sheet": "Sheet 1", "header": 9},
    "IPI":  {"file_key": "ipi",  "sheet": "Sheet 1", "header": 10},
}

for label, cfg in wide_format_sources.items():
    path = RAW_FILES[cfg["file_key"]]
    df_wide = pd.read_excel(path, sheet_name=cfg["sheet"], header=cfg["header"])

    date_cols = [c for c in df_wide.columns if str(c).strip() not in ("TIME", "GEO (Labels)") and str(c) != "nan"]

    date_range_results.append({
        "variable": label,
        "country": "All 4 (same columns)",
        "start": date_cols[0] if date_cols else None,
        "end": date_cols[-1] if date_cols else None,
    })

# ------------------------------------------------------------
# RESULTS TABLE
# ------------------------------------------------------------
date_range_df = pd.DataFrame(date_range_results)
print(f"Date ranges checked: {len(date_range_df)} entries\n")
date_range_df

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Date ranges checked: 9 entries



/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,variable,country,start,end
0,CLIFS,N/A (single series),1970-01-31 00:00:00,2026-04-30 00:00:00
1,Energy (Brent/Gas),N/A (single series),1960-01-01 00:00:00,2026-05-01 00:00:00
2,GPR,Germany,1997-01-01 00:00:00,2026-04-01 00:00:00
3,GPR,France,1985-01-01 00:00:00,2026-04-01 00:00:00
4,GPR,Italy,1997-01-01 00:00:00,2026-04-01 00:00:00
5,GPR,Spain,1996-01-01 00:00:00,2026-04-01 00:00:00
6,HICP,All 4 (same columns),1996-01,2025-12
7,GDP,All 4 (same columns),1990-Q1,2026-Q1
8,IPI,All 4 (same columns),1990-01,2026-04


In [26]:
# ============================================================
# COMPUTE THE COMMON DATE INTERVAL ACROSS ALL SOURCES
# ============================================================
# Diagnostic: identify the intersection of raw source coverage.
# The estimation sample is determined after cleaning, aggregation and transformations.

import re

def normalize_to_timestamp(value, is_start=True):
    """
    Converts the different date formats found in date_range_df
    (Timestamp, 'YYYY-Qx', 'YYYY-MM') into a single comparable Timestamp.
    is_start=True snaps to the first day of the period; False snaps to the last.
    """
    if isinstance(value, pd.Timestamp):
        return value

    value = str(value)

    # Format "YYYY-Qx" (e.g. GDP: "1990-Q1")
    quarter_match = re.match(r"(\d{4})-Q([1-4])", value)
    if quarter_match:
        year, quarter = int(quarter_match.group(1)), int(quarter_match.group(2))
        month = (quarter - 1) * 3 + 1
        ts = pd.Timestamp(year=year, month=month, day=1)
        return ts if is_start else ts + pd.offsets.QuarterEnd(0)

    # Format "YYYY-MM" (e.g. HICP: "1996-01", IPI: "1990-01")
    month_match = re.match(r"(\d{4})-(\d{2})", value)
    if month_match:
        year, month = int(month_match.group(1)), int(month_match.group(2))
        ts = pd.Timestamp(year=year, month=month, day=1)
        return ts if is_start else ts + pd.offsets.MonthEnd(0)

    return pd.NaT

# Apply normalization to the existing date_range_df
date_range_df["start_norm"] = date_range_df["start"].apply(lambda v: normalize_to_timestamp(v, is_start=True))
date_range_df["end_norm"]   = date_range_df["end"].apply(lambda v: normalize_to_timestamp(v, is_start=False))

# The binding constraint: latest start, earliest end
common_start = date_range_df["start_norm"].max()
common_end   = date_range_df["end_norm"].min()

binding_start_var = date_range_df.loc[date_range_df["start_norm"] == common_start, ["variable", "country"]]
binding_end_var   = date_range_df.loc[date_range_df["end_norm"] == common_end, ["variable", "country"]]

print(f"Common usable interval: {common_start.to_period('Q')} to {common_end.to_period('Q')}")
print(f"\nBinding start variable(s):")
print(binding_start_var.to_string(index=False))
print(f"\nBinding end variable(s):")
print(binding_end_var.to_string(index=False))

Common usable interval: 1997Q1 to 2025Q4

Binding start variable(s):
variable country
     GPR Germany
     GPR   Italy

Binding end variable(s):
variable              country
    HICP All 4 (same columns)


In [27]:
# ============================================================
# D1. CREATE QUARTERLY SKELETON
# ============================================================
# 1997Q1 is retained so notebook 02 can construct first log differences.
# The final transformed estimation sample begins in 1997Q2 and ends in 2025Q4.

SKELETON_START = "1997Q1"
SKELETON_END = "2025Q4"

quarters = pd.period_range(start=SKELETON_START, end=SKELETON_END, freq="Q")

master_skeleton = pd.DataFrame({
    "quarter": quarters.astype(str),
    "year": quarters.year,
    "q": quarters.quarter,
})

assert master_skeleton["quarter"].iloc[0] == SKELETON_START
assert master_skeleton["quarter"].iloc[-1] == SKELETON_END

print(
    f"Quarterly skeleton: {len(master_skeleton)} observations "
    f"({SKELETON_START} – {SKELETON_END})"
)
master_skeleton


Quarterly skeleton: 116 observations (1997Q1 – 2025Q4)


,quarter,year,q
0,1997Q1,1997,1
1,1997Q2,1997,2
2,1997Q3,1997,3
3,1997Q4,1997,4
4,1998Q1,1998,1
...,...,...,...
111,2024Q4,2024,4
112,2025Q1,2025,1
113,2025Q2,2025,2
114,2025Q3,2025,3


In [28]:
# Save skeleton
master_skeleton.to_excel(SKELETON_PATH, index=False)
print(f"Quarterly skeleton saved to: {SKELETON_PATH}")


Quarterly skeleton saved to: /content/drive/MyDrive/TFM_GitHub/data/processed/02_master_quarterly_skeleton.xlsx


## Block E — Final Output Check

In [29]:
# ============================================================
# E1. VERIFY EXPECTED OUTPUTS
# ============================================================

EXPECTED_OUTPUTS = {
    "import_decisions": IMPORT_DECISIONS_PATH,
    "master_quarterly_skeleton": SKELETON_PATH,
}

output_check = pd.DataFrame([
    {"output_key": key, "output_path": str(path), "exists": path.exists()}
    for key, path in EXPECTED_OUTPUTS.items()
])

if output_check["exists"].all():
    print("All expected outputs are present. Ready for notebook 02.")
else:
    print("Some expected outputs are missing:")
    print(output_check[~output_check["exists"]].to_string(index=False))

output_check


All expected outputs are present. Ready for notebook 02.


,output_key,output_path,exists
0,import_decisions,/content/drive/MyDrive/TFM_GitHub/data/process...,True
1,master_quarterly_skeleton,/content/drive/MyDrive/TFM_GitHub/data/process...,True
